# Summit-Sim: Complete E2E Workflow Demo

Demonstrates the full Summit-Sim workflow:
1. **Phase 0: Teacher Review** - Teacher generates and approves scenario
2. **Phase 1: Scenario Generation** - AI generates wilderness rescue scenario
3. **Phase 2: Simulation** - Student navigates scenario with human-in-the-loop
4. **Phase 3: Debrief** - Performance analysis and learning report

All phases are traced to MLflow for visibility.

## 1. Setup & Imports

In [1]:
import warnings

# Suppress autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

import mlflow

from summit_sim.graphs import create_student_graph, create_teacher_graph
from summit_sim.schemas import TeacherConfig, generate_scenario_id
from summit_sim.settings import settings

# Initialize MLflow
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
mlflow.pydantic_ai.autolog()
mlflow.langchain.autolog(run_tracer_inline=True)

print(f"✓ MLflow: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key: {bool(settings.openrouter_api_key)}")

✓ MLflow: https://mlflow.bhamm-lab.com
✓ Experiment: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key: True


## 2. Phase 0: Teacher Review Flow

Demonstrates the teacher workflow:
1. **Configuration** - Teacher sets scenario parameters
2. **Generation** - AI creates scenario from config
3. **Review** - Teacher reviews scenario via interrupt
4. **Approval** - Teacher approves scenario
5. **Share** - Unique URL generated for students

In [2]:
# Teacher configuration
teacher_config = TeacherConfig(
    num_participants=3,
    activity_type="hiking",
    difficulty="med",
    class_id="demo-class-2024",
)

print(
    f"Teacher Config: {teacher_config.activity_type} scenario for {teacher_config.num_participants} participants"
)
print(f"Difficulty: {teacher_config.difficulty}")
print(f"Class ID: {teacher_config.class_id}")

Teacher Config: hiking scenario for 3 participants
Difficulty: med
Class ID: demo-class-2024


In [3]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create teacher review graph with memory
memory = InMemorySaver()
teacher_graph = create_teacher_graph(checkpointer=memory)

# Initialize state
initial_teacher_state = {
    "teacher_config": teacher_config,
}

print("✓ Teacher review graph initialized")

✓ Teacher review graph initialized


In [4]:
# Run teacher review graph with MLflow session tracking
import uuid

import mlflow

# Generate a scenario ID for this session
scenario_id = generate_scenario_id()
session_id = str(uuid.uuid4())
graph_config = {"configurable": {"thread_id": session_id}}

run_name = f"gen-{teacher_config.activity_type}-{teacher_config.num_participants}p-{teacher_config.difficulty}"

print(f"\n📋 Session ID: {session_id}")
print("🎮 Starting teacher review...\n")
print("=" * 60)
print("TEACHER REVIEW FLOW")
print("=" * 60 + "\n")

# Run to interrupt point (scenario generation + review)
result = await teacher_graph.ainvoke(initial_teacher_state, graph_config)

# Display generated scenario for review
scenario = result["scenario_draft"]
print(f"\n📋 Generated Scenario: {scenario['title']}")
print(f"Setting: {scenario['setting']}")
print(f"Patient: {scenario['patient_summary']}")
print(f"Turns: {len(scenario['turns'])}")
print("\nLearning Objectives:")
for obj in scenario["learning_objectives"]:
    print(f"  • {obj}")

# Simulate teacher approval
print("\n" + "-" * 60)
print("Teacher reviews scenario and clicks 'Approve'...")
print("-" * 60 + "\n")

# Resume with approval
final_result = await teacher_graph.ainvoke(
    Command(resume={"decision": "approve"}), graph_config
)

# Display shareable URL
class_id = final_result["class_id"]
shareable_url = f"/scenario/{scenario_id}?class_id={class_id}"

print("\n" + "=" * 60)
print("✅ SCENARIO APPROVED")
print("=" * 60)
print(f"\n📍 Scenario ID: {scenario_id}")
print(f"👥 Class ID: {class_id}")
print(f"\n🔗 Shareable URL: {shareable_url}")
print("\n✓ Ready for students to join!")

# Store for next phase
approved_scenario = scenario
approved_scenario_id = scenario_id
approved_class_id = class_id


📋 Session ID: 0b5560f3-c227-4bd6-8273-50fcf75aad4c
🎮 Starting teacher review...

TEACHER REVIEW FLOW



2026/03/25 10:40:55 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f2188323100> was created in a different Context
2026/03/25 10:41:06 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f21880a3880> was created in a different Context
2026/03/25 10:41:06 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f218a0a8a80> was created in a different Context
2026/03/25 10:41:06 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f21881ebdc0> was created in a different Context
2026/03/25 10:41:06 WARNING mlflow.tracing.f


📋 Generated Scenario: The Ridge Line Emergency
Setting: A high-altitude, exposed rocky ridgeline, approximately 8,000 ft elevation. A severe thunderstorm is rapidly approaching from the west. Mid-afternoon.
Patient: Mark, 34 years old, fit hiker. Sustained a 10-foot fall down a scree slope. He is conscious but dazed. Visible deformity in the right ankle.
Turns: 4

Learning Objectives:
  • Prioritize scene safety over patient assessment during rapid environmental changes (the thunderstorm).
  • Execute a systematic patient assessment (SAMPLE/head-to-toe) while managing environmental hazards.
  • Apply proper immobilization for a lower limb fracture and management of potential head injury.

------------------------------------------------------------
Teacher reviews scenario and clicks 'Approve'...
------------------------------------------------------------


✅ SCENARIO APPROVED

📍 Scenario ID: scn-25c49bc7
👥 Class ID: 72956b

🔗 Shareable URL: /scenario/scn-25c49bc7?class_id=72956b

✓ 

Trace(trace_id=tr-2bacb136a4b7ae920a0696b7d385f62f)

## 3. Phase 1: Scenario Generation (Approved)

The approved scenario from Phase 0 is now ready for students.

In [6]:
# Use the approved scenario from Phase 0
scenario = approved_scenario
scenario_id = approved_scenario_id
config = teacher_config

print(f"Using approved scenario: {scenario['title']}")
print(f"Scenario ID: {scenario_id}")
print(f"Class ID: {approved_class_id}")
print("\n✓ Ready for Phase 2: Simulation")

Using approved scenario: The Ridge Line Emergency
Scenario ID: scn-25c49bc7
Class ID: 72956b

✓ Ready for Phase 2: Simulation


## 3. Phase 2: Simulation

Run the interactive simulation with human-in-the-loop.
The graph will pause at each turn for you to make a choice.

In [7]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create graph with memory
memory = InMemorySaver()
graph = create_student_graph(checkpointer=memory)

# Initialize state
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario["starting_turn_id"],
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
    "debrief_report": None,
    "mlflow_run_id": "",
}

print("✓ Graph initialized")
print(f"✓ Starting at turn {initial_state['current_turn_id']}")

✓ Graph initialized
✓ Starting at turn 0


In [9]:
# Run simulation with MLflow session tracking
import uuid

session_id = str(uuid.uuid4())
graph_config = {"configurable": {"thread_id": session_id}}

run_name = f"sim-{config.activity_type}-{config.num_participants}p-{config.difficulty}"

with mlflow.start_run(run_name=run_name):
    print(f"\n📋 Session ID: {session_id}")
    print("🎮 Starting simulation...\n")
    print("=" * 60)
    print("INTERACTIVE SIMULATION")
    print("=" * 60 + "\n")

    current_state = initial_state
    turn_count = 0
    simulation_complete = False

    print("\n=== STARTING SIMULATION ===\n")

    while not simulation_complete:
        input_state = initial_state if turn_count == 0 else None
        turn_count += 1
        print(f"--- TURN {turn_count} ---")

        async for event in graph.astream(input_state, graph_config):
            if "__interrupt__" in event:
                interrupt_obj = event["__interrupt__"]
                if hasattr(interrupt_obj, "value"):
                    interrupt_data = interrupt_obj.value
                elif isinstance(interrupt_obj, (list, tuple)):
                    interrupt_data = (
                        interrupt_obj[0].value
                        if hasattr(interrupt_obj[0], "value")
                        else interrupt_obj[0]
                    )
                else:
                    interrupt_data = interrupt_obj

                print(f"\n📖 {interrupt_data['narrative']}\n")
                print("Choices:")
                for i, choice in enumerate(interrupt_data["choices"], 1):
                    print(f"  {i}. {choice['description']}")

                while True:
                    try:
                        sel = int(input("\nSelect: ")) - 1
                        if 0 <= sel < len(interrupt_data["choices"]):
                            break
                        print("Invalid")
                    except ValueError:
                        print("Enter number")

                selected = interrupt_data["choices"][sel]
                current_state = await graph.ainvoke(
                    Command(resume={"choice_id": selected["choice_id"]}), graph_config
                )
                break

            # Check for graph completion
            if event == {}:
                simulation_complete = True
                break

        # Check if state indicates completion
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("Safety stop")
            break

    print("\n=== SIMULATION COMPLETE ===")

    # Log final results to MLflow
    debrief = current_state.get("debrief_report")
    mlflow.log_metrics(
        {
            "total_turns": len(current_state["transcript"]),
            "is_complete": float(current_state["is_complete"]),
            "learning_moments_count": len(current_state["key_learning_moments"]),
            **({"final_score": debrief["final_score"]} if debrief else {}),
        }
    )
    if debrief:
        mlflow.set_tag("pass_fail", debrief["completion_status"])
    print("\n✓ Results logged to MLflow")
    print(f"✓ Total turns: {len(current_state['transcript'])}")
    print(f"✓ Learning moments: {len(current_state['key_learning_moments'])}")

2026/03/25 10:44:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f218210c580> was created in a different Context
2026/03/25 10:44:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f218210ee00> was created in a different Context



📋 Session ID: df1f4d40-3bef-4ec3-957e-e423553e7fd5
🎮 Starting simulation...

INTERACTIVE SIMULATION


=== STARTING SIMULATION ===

--- TURN 1 ---

📖 You are hiking the exposed ridgeline with two partners when Mark trips, falls 10 feet down a rocky scree slope, and lands awkwardly. A thunderstorm is rapidly approaching with lightning strikes visible less than a mile away. Mark is conscious but dazed, clutching his right ankle, which looks clearly deformed. Thunder is booming nearby.

Choices:
  1. Immediately move Mark to a more protected area just off the ridge before assessing the injury.
  2. Stay where you are and perform a full SAMPLE assessment and head-to-toe check immediately.
  3. Immediately attempt to splint the ankle while exposed on the ridge before moving.


2026/03/25 10:44:22 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f21820e2a00> was created in a different Context
2026/03/25 10:44:35 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f218220a900> was created in a different Context
2026/03/25 10:44:35 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f2182325780> was created in a different Context
2026/03/25 10:44:35 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f21820e2a00> was created in a different Context
2026/03/25 10:44:35 WARNING mlflow.utils.aut

--- TURN 2 ---


2026/03/25 10:44:36 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f21820e21c0> was created in a different Context
2026/03/25 10:44:36 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f2182182680> was created in a different Context


...

--- TURN 3 ---

📖 The storm is passing. Mark is stable but shows signs of a concussion (headache, confusion). The ankle is clearly fractures. You have the necessary materials to splint and insulate him. What is your final plan?

Choices:
  1. Splint the ankle, initiate active warming for heat loss, monitor mental status, and wait for the storm to pass before attempting self-extraction or signaling for rescue.
  2. Attempt an immediate, rough carrying extraction down the steep, wet slope to reach safety faster.


Deserializing unregistered type summit_sim.graphs.shared.TranscriptEntry from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.graphs.shared', 'TranscriptEntry')]
Deserializing unregistered type summit_sim.graphs.shared.TranscriptEntry from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.graphs.shared', 'TranscriptEntry')]
2026/03/25 10:44:54 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f218292f080> was created in a different Context
2026/03/25 10:45:02 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f2182359700> was created in a different Context
2026/03/25 10:45:02 WARNING mlflow.utils.autologging_utils: Encountered 


=== SIMULATION COMPLETE ===

✓ Results logged to MLflow
✓ Total turns: 3
✓ Learning moments: 6
🏃 View run sim-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/b6d43978afa448d4b85249e747a6814f
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-97a603e0fa0338410e4629243a291d97), Trace(trace_id=tr-ce64c1ca322c13d335ce40f01dec10f9)]

2026/03/25 10:45:12 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f21bf914cc0> at 0x7f21820a1940> was created in a different Context


## 4. Phase 3: Debrief

View the comprehensive performance report generated automatically by the simulation graph.

In [10]:
debrief_report = current_state["debrief_report"]

print("=" * 60)
print("DEBRIEF REPORT")
print("=" * 60)

print(f"\n📊 Final Score: {debrief_report['final_score']:.1f}%")
status_emoji = "✅" if debrief_report["completion_status"] == "pass" else "❌"
print(f"{status_emoji} Status: {debrief_report['completion_status'].upper()}")

print("\n📝 Summary:")
print(debrief_report["summary"])

if debrief_report["key_mistakes"]:
    print("\n⚠️ Key Mistakes:")
    for mistake in debrief_report["key_mistakes"]:
        print(f"  • {mistake}")

if debrief_report["strong_actions"]:
    print("\n💪 Strong Actions:")
    for action in debrief_report["strong_actions"]:
        print(f"  • {action}")

if debrief_report["teaching_points"]:
    print("\n📚 Teaching Points:")
    for point in debrief_report["teaching_points"]:
        print(f"  • {point}")

if debrief_report["best_next_actions"]:
    print("\n🎯 Recommendations:")
    for rec in debrief_report["best_next_actions"]:
        print(f"  • {rec}")

DEBRIEF REPORT

📊 Final Score: 66.7%
❌ Status: FAIL

📝 Summary:
You demonstrated a strong foundational understanding of scene safety and effective decision-making regarding environmental risks. However, you struggled with 'tunnel vision,' where you fixated on the visible ankle injury at the expense of a thorough assessment of the patient's systemic status. By missing the signs of a potential concussion and hypothermia early on, you risked delaying critical care for a potentially life-threatening condition. With a more systematic approach to your secondary assessment, your patient outcomes will improve significantly.

⚠️ Key Mistakes:
  • Focused exclusively on the local ankle injury, missing the systemic indicators of head injury (slurred speech) and hypothermia (shivering).
  • Failed to recognize that a visible injury often acts as a distraction from underlying life threats during the secondary survey.

💪 Strong Actions:
  • Correctly prioritized scene safety by moving the patient aw

Trace(trace_id=tr-48b6971dfc25c7210eeca16ea642eebb)

## 5. Results Summary

Complete transcript and learning moments from the simulation.

In [14]:
print("\n" + "=" * 60)
print("COMPLETE TRANSCRIPT")
print("=" * 60 + "\n")

for i, entry in enumerate(current_state["transcript"], 1):
    status = "✓" if entry.was_correct else "✗"
    print(f"Turn {i} {status}")
    print(f"  Situation: {entry.turn_narrative[:80]}...")
    print(f"  Choice: {entry.choice_description}")
    print(f"  Feedback: {entry.feedback[:100]}...")
    print()


COMPLETE TRANSCRIPT

Turn 1 ✓
  Situation: You are hiking the exposed ridgeline with two partners when Mark trips, falls 10...
  Choice: Immediately move Mark to a more protected area just off the ridge before assessing the injury.
  Feedback: Excellent decision. Lightning strikes on an exposed ridgeline are a lethal and immediate threat; by ...

Turn 2 ✗
  Situation: You have successfully moved Mark to a slightly more protected ledge beneath a sm...
  Choice: Focus exclusively on splinting the ankle to make him comfortable.
  Feedback: While it's natural to want to fix a visible injury like an ankle fracture, focusing solely on it cau...

Turn 3 ✓
  Situation: The storm is passing. Mark is stable but shows signs of a concussion (headache, ...
  Choice: Splint the ankle, initiate active warming for heat loss, monitor mental status, and wait for the storm to pass before attempting self-extraction or signaling for rescue.
  Feedback: Excellent judgment. Choosing to shelter in place prot

In [15]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{i}. {moment}")


KEY LEARNING MOMENTS

1. Always assess for immediate environmental hazards before starting a medical exam; your safety as a rescuer is paramount.
2. Remember the principle of 'Life over Limb': a broken ankle is serious, but a lightning strike or rapid-onset hypothermia are immediate life threats that take precedence.
3. Always prioritize systemic life threats—such as mental status and thermoregulation—before addressing localized musculoskeletal injuries.
4. A comprehensive patient assessment (like SAMPLE) helps identify hidden, potentially life-threatening conditions like a traumatic brain injury that are more critical than a fracture.
5. Prioritizing environmental safety during a weather event is the first step of rescue; you cannot treat a patient effectively if you become a casualty yourself.
6. In cases of concussion and incipient hypothermia, stabilization and insulation must happen before any movement, as mobility is degraded and the patient is highly vulnerable during transport

---
## ✅ Complete Workflow Demo

All phases executed:
- ✓ Scenario generated with AI
- ✓ Simulation completed with human-in-the-loop
- ✓ Debrief report generated
- ✓ All traces logged to MLflow

View traces in MLflow UI at your configured tracking URI.